# Chapter 5. Geographically Weighted Regression

*Starting With What You Have: A Quantitative Field Guide for Urban Research in Data-Scarce Settings*

Runs in a browser with no installation. Open in Google Colab and choose Runtime, then Run all.


## Step 0. Install the packages

`mgwr` is not on conda-forge and comes from pip. Locally, install geopandas from conda-forge **first**, then add mgwr.

In [ ]:
!pip install -q mgwr geopandas mapclassify

## Step 1. Load the data and check it

Bandwidth is measured in map units, so a projected CRS is essential. In latitude and longitude you get uninterpretable values such as a bandwidth of 0.02 degrees.

In [ ]:
import geopandas as gpd
import numpy as np

gdf = gpd.read_file("data/phnompenh_parcels.geojson")
Y_COL = "land_price"
X_COLS = ["dist_cbd", "road_access", "flood_risk"]

print("observations:", len(gdf), "| at least 100?", len(gdf) >= 100)
print("CRS projected?", not gdf.crs.is_geographic)

## Step 2. Standardisation, mandatory if MGWR is planned

Bandwidths can only be compared when variables share a scale.

In [ ]:
y = gdf[Y_COL].values.reshape(-1, 1)
X = gdf[X_COLS].values
coords = list(zip(gdf.geometry.x, gdf.geometry.y))

y = (y - y.mean()) / y.std()
X = (X - X.mean(axis=0)) / X.std(axis=0)
print("standardised. y mean:", round(float(y.mean()), 6))

## Step 3. Run OLS first, to create a baseline

Without a benchmark there is no way to judge whether GWR is genuinely better.

In [ ]:
Xc = np.hstack([np.ones((X.shape[0], 1)), X])
beta, *_ = np.linalg.lstsq(Xc, y, rcond=None)
resid = y - Xc @ beta
r2 = 1 - (resid ** 2).sum() / ((y - y.mean()) ** 2).sum()
print(f"OLS R2 = {float(r2):.3f}")

## Step 4. Search for the bandwidth

`fixed=False` selects the adaptive kernel, fixing the number of neighbours rather than a radius. Urban data is dense in the centre and sparse at the edge, so adaptive is almost always right.

In [ ]:
from mgwr.sel_bw import Sel_BW

bw = Sel_BW(coords, y, X, fixed=False).search()
print("adaptive bandwidth =", bw, "nearest neighbours")

## Step 5. Fit GWR

Compare against the OLS baseline. A large improvement is itself evidence that the relationship varies across space.

In [ ]:
from mgwr.gwr import GWR

gwr_model = GWR(coords, y, X, bw, fixed=False).fit()
print(f"GWR R2 = {gwr_model.R2:.3f}   AICc = {gwr_model.aicc:.1f}")

## Step 6. Did GWR recover the truth?

The distance-to-centre coefficient was planted deliberately: strong near the centre, near zero at the periphery. This check is impossible in real research, which is why practising on synthetic data first is worth the effort.

In [ ]:
for i, name in enumerate(["intercept"] + X_COLS):
    gdf["b_" + name] = gwr_model.params[:, i]

r = np.corrcoef(gdf["b_dist_cbd"], gdf["true_b_dist"])[0, 1]
print(f"estimated local coefficients vs planted truth, r = {r:.3f}")

## Step 7. MGWR: the spatial scale of each variable

A narrow bandwidth indicates a local process, a wide one a near-global process. Select between models on **AICc, not R-squared**.

In [ ]:
from mgwr.gwr import MGWR

selector = Sel_BW(coords, y, X, multi=True)
bws = selector.search(multi_bw_min=[2])
mgwr_model = MGWR(coords, y, X, selector).fit()

for name, b in zip(["intercept"] + X_COLS, bws):
    kind = "local" if b < 100 else "near-global"
    print(f"  {name:14s} bandwidth {b:6.0f}   {kind}")
print(f"MGWR AICc = {mgwr_model.aicc:.1f}   (GWR AICc = {gwr_model.aicc:.1f}; lower is better)")

## Step 8. Map the local coefficients

One surface varies sharply across the city; the other is close to flat.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
gdf.plot(column="b_dist_cbd", cmap="RdBu_r", legend=True, ax=ax[0], markersize=14)
ax[0].set_title("Distance to centre: varies sharply")
gdf.plot(column="b_road_access", cmap="RdBu_r", legend=True, ax=ax[1], markersize=14)
ax[1].set_title("Road access: near uniform")
for a in ax:
    a.set_axis_off()
plt.tight_layout()
plt.show()

## Step 9. Significance testing, a gate you must pass through

Because a test is repeated at every point, the critical t value is corrected and is considerably stricter than 1.96. Where a coefficient is not significant, the correct wording is not that there is no effect but that there is insufficient evidence of one.

In [ ]:
crit = float(gwr_model.critical_tval())
share = float((np.abs(gwr_model.tvalues[:, 1]) > crit).mean())
print(f"corrected critical t = {crit:.3f}")
print(f"share of observations where distance-to-centre is significant: {share:.1%}")

---

**What to do next.** Compare against Section 5.4. With fewer than 100 observations, use Chapter 4 instead: local coefficients estimated from too few neighbours are unstable.